# ETL-16 · Nulos e categorias

- **Etapa:** ETL-16, Nulos e categorias
- **Responsável:** Pedro Wellington
- **Issue:** ETL-16
- **Entrada:** `data/pessoas_*.csv`
- **Saída:** `intermediarios/<arquivo>.parquet`

## 1. Setup

In [15]:
from pathlib import Path
import pandas as pd

# Raiz do repositório: sobe de pasta em pasta até achar a pasta .git
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())

DADOS  = RAIZ / "data"                         # CSVs brutos (estão no Git)
INTERM = DADOS / "intermediarios"              # .parquet (o .gitignore já ignora)
INTERM.mkdir(parents=True, exist_ok=True)

SAIDA = INTERM / "05_nulos_categorias.parquet"  # resultado desta etapa

## 2. Carregar a entrada

In [16]:
# lê os 36 CSVs (2021 a 2023) e junta numa tabela só
arquivos = sorted(DADOS.glob("pessoas_*.csv"))

lista = []
for a in arquivos:
    d = pd.read_csv(a, encoding='latin-1', sep=';')
    d['ano_origem'] = int(a.stem[-4:])    # ano vem do nome do arquivo
    d['arquivo_origem'] = a.name          # nome do arquivo de origem
    lista.append(d)

df = pd.concat(lista, ignore_index=True)
print(len(arquivos), 'arquivos')
print('shape:', df.shape)
df.head()

36 arquivos
shape: (806977, 33)


,id_sinistro,id_pessoa,id_veiculo,cod_ibge,municipio,regiao_administrativa,tipo_via,tipo_veiculo_vitima,tipo_de_vitima,modo_transporte_vitima,...,data_obito,ano_obito,mes_obito,dia_obito,ano_mes_obito,local_obito,local_via,tempo_sinistro_obito,ano_origem,arquivo_origem
0,2240338,1979797,2874666.0,3552502,SUZANO,METROPOLITANA DE SÃO PAULO,VIAS URBANAS,AUTOMOVEL,PASSAGEIRO,AUTOMOVEL,...,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,2021,pessoas_01-2021.csv
1,2240338,1979798,2874666.0,3552502,SUZANO,METROPOLITANA DE SÃO PAULO,VIAS URBANAS,AUTOMOVEL,CONDUTOR,AUTOMOVEL,...,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,2021,pessoas_01-2021.csv
2,1447800,1284984,NaN,3550308,SAO PAULO,METROPOLITANA DE SÃO PAULO,VIAS URBANAS,NaN,PEDESTRE,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,2021,pessoas_01-2021.csv
3,1783410,563279,2648403.0,3550308,SAO PAULO,METROPOLITANA DE SÃO PAULO,VIAS URBANAS,AUTOMOVEL,CONDUTOR,AUTOMOVEL,...,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,2021,pessoas_01-2021.csv
4,1798856,687322,NaN,3547601,SANTA ROSA DE VITERBO,RIBEIRÃO PRETO,VIAS URBANAS,NaN,PASSAGEIRO,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,2021,pessoas_01-2021.csv


## 3. Diagnóstico ANTES

In [17]:
def diagnostico(df: pd.DataFrame) -> pd.DataFrame:
    """Tabela-resumo usada antes e depois: tipo, % de nulos e nº de valores distintos por coluna."""
    return pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'pct_nulo': (df.isna().mean() * 100).round(2),
        'n_distintos': df.nunique(),
    })

antes = diagnostico(df)
antes

,dtype,pct_nulo,n_distintos
id_sinistro,int64,0.00,402695
id_pessoa,int64,0.00,806977
id_veiculo,float64,21.04,554294
cod_ibge,int64,0.00,645
municipio,str,0.00,645
regiao_administrativa,str,0.00,16
tipo_via,str,0.00,3
tipo_veiculo_vitima,str,20.98,7
tipo_de_vitima,str,0.00,4
modo_transporte_vitima,str,20.54,8


### 4.1 Marcar quem morreu
Cria a coluna `flag_obito`: True se `data_obito` está preenchida. Resultado esperado: 15.793 óbitos.

In [18]:
def criar_flag_obito(df: pd.DataFrame) -> pd.DataFrame:
    """Cria flag_obito: True se data_obito está preenchida (a pessoa morreu)."""
    df = df.copy()
    df['flag_obito'] = df['data_obito'].notna()
    return df

df = criar_flag_obito(df)
print(df['flag_obito'].value_counts())

flag_obito
False    791184
True      15793
Name: count, dtype: int64


### 4.2 Testar a hipótese dos pedestres
Entre as linhas sem veículo, quantas são de cada tipo de vítima, por ano. Só consulta, não altera nada.

In [19]:
# linhas sem veículo
sem_veiculo = df[df['tipo_veiculo_vitima'].isna()]

# quantas de cada tipo de vítima, por ano
print(pd.crosstab(sem_veiculo['ano_origem'], sem_veiculo['tipo_de_vitima'], margins=True))

# a mesma tabela, em % de cada ano
print((pd.crosstab(sem_veiculo['ano_origem'], sem_veiculo['tipo_de_vitima'], normalize='index') * 100).round(2))

tipo_de_vitima  CONDUTOR  NAO DISPONIVEL  PASSAGEIRO  PEDESTRE     All
ano_origem                                                            
2021                1134           32779        6811     14468   55192
2022                1324           28653        7108     16181   53266
2023                1164           37072        7083     15524   60843
All                 3622           98504       21002     46173  169301
tipo_de_vitima  CONDUTOR  NAO DISPONIVEL  PASSAGEIRO  PEDESTRE
ano_origem                                                    
2021                2.05           59.39       12.34     26.21
2022                2.49           53.79       13.34     30.38
2023                1.91           60.93       11.64     25.51


### 4.3 Nulos de veículo
Pedestre sem veículo vira `NÃO SE APLICA`. Condutor, passageiro e tipo desconhecido sem veículo viram `NÃO INFORMADO`.

In [20]:
def tratar_nulos_veiculo(df: pd.DataFrame) -> pd.DataFrame:
    """Veículo nulo: pedestre vira NÃO SE APLICA; os demais viram NÃO INFORMADO."""
    df = df.copy()
    pedestre = df['tipo_de_vitima'] == 'PEDESTRE'
    for col in ['tipo_veiculo_vitima', 'modo_transporte_vitima']:
        nulo = df[col].isna()
        df.loc[nulo & pedestre, col] = 'NÃO SE APLICA'
        df.loc[nulo & ~pedestre, col] = 'NÃO INFORMADO'
    return df

df = tratar_nulos_veiculo(df)
print(df['tipo_veiculo_vitima'].value_counts(dropna=False))

tipo_veiculo_vitima
AUTOMOVEL         315274
MOTOCICLETA       274658
NÃO INFORMADO     123128
NÃO SE APLICA      46173
CAMINHAO           22332
ONIBUS             16336
OUTROS              7472
BICICLETA           1124
NAO DISPONIVEL       480
Name: count, dtype: int64


### 4.4 Conferir as idades
Verifica mínimo, máximo e valores fora de 0 a 110. Só consulta: nenhuma idade inválida foi encontrada.

In [21]:
# resumo das idades, por ano
print(df.groupby('ano_origem')['idade'].describe())

# quantas idades fora do intervalo plausível (0 a 110)
invalidas = df[(df['idade'] < 0) | (df['idade'] > 110)]
print(len(invalidas), 'idades fora de 0 a 110')
print(invalidas.groupby('ano_origem').size())

               count       mean        std  min   25%   50%   75%    max
ano_origem                                                              
2021        257411.0  36.713528  14.778949  0.0  25.0  34.0  46.0  102.0
2022        262331.0  37.416268  15.113480  0.0  25.0  35.0  47.0   99.0
2023        278846.0  37.671173  15.292993  0.0  25.0  36.0  48.0  100.0
0 idades fora de 0 a 110
Series([], dtype: int64)


### 4.5 Nulos de texto
`profissao`, `nacionalidade` e `local_via` vazios viram `NÃO INFORMADO`. `local_obito` vazio vira `NÃO SE APLICA` para quem não morreu.

In [22]:
def tratar_nulos_texto(df: pd.DataFrame) -> pd.DataFrame:
    """local_obito vazio de quem não morreu = NÃO SE APLICA; os demais nulos de texto = NÃO INFORMADO."""
    df = df.copy()

    # quem não morreu não tem local de óbito
    nulo = df['local_obito'].isna()
    df.loc[nulo & ~df['flag_obito'], 'local_obito'] = 'NÃO SE APLICA'
    df.loc[nulo & df['flag_obito'], 'local_obito'] = 'NÃO INFORMADO'

    # campos que deviam ter valor e vieram vazios
    for col in ['profissao', 'nacionalidade', 'local_via']:
        df[col] = df[col].fillna('NÃO INFORMADO')
    return df

df = tratar_nulos_texto(df)

# o que ainda tem nulo
nulos = df.isna().sum()
print(nulos[nulos > 0])

id_veiculo              169754
idade                     8389
grau_de_instrucao       632289
data_obito              791184
ano_obito               791184
mes_obito               791184
dia_obito               791184
ano_mes_obito           791184
tempo_sinistro_obito    791184
dtype: int64


### 4.6 Números sem casa decimal
Troca `float` por `Int64` (inteiro que aceita nulo) em idade, `id_veiculo` e partes da data de óbito.

In [23]:
def converter_inteiros(df: pd.DataFrame) -> pd.DataFrame:
    """Troca float por Int64 (inteiro que aceita nulo): idade, id_veiculo e partes da data de óbito."""
    df = df.copy()
    for col in ['idade', 'id_veiculo', 'ano_obito', 'mes_obito', 'dia_obito']:
        df[col] = df[col].astype('Int64')
    return df

df = converter_inteiros(df)
print(df[['idade', 'id_veiculo', 'ano_obito', 'mes_obito', 'dia_obito']].dtypes)
df[['idade', 'id_veiculo']].head()

idade         Int64
id_veiculo    Int64
ano_obito     Int64
mes_obito     Int64
dia_obito     Int64
dtype: object


,idade,id_veiculo
0,31,2874666
1,34,2874666
2,65,<NA>
3,26,2648403
4,66,<NA>


### 4.7 Grau de instrução
Primeiro vê o % preenchido mês a mês (sempre ~20% a 23%, não existe período confiável). Depois os nulos viram `NÃO INFORMADO`.

In [24]:
# % de grau_de_instrucao preenchido, mês a mês (36 meses)
tabela = df.groupby('ano_mes_sinistro')['grau_de_instrucao'].apply(lambda s: round(s.notna().mean() * 100, 2))
print(tabela.to_string())

ano_mes_sinistro
2021/01    22.62
2021/02    21.58
2021/03    20.99
2021/04    21.32
2021/05    21.62
2021/06    20.90
2021/07    20.92
2021/08    21.61
2021/09    22.07
2021/10    22.19
2021/11    22.10
2021/12    22.53
2022/01    22.75
2022/02    21.98
2022/03    21.98
2022/04    22.75
2022/05    21.85
2022/06    21.97
2022/07    22.81
2022/08    21.52
2022/09    21.41
2022/10    22.22
2022/11    21.54
2022/12    23.59
2023/01    22.63
2023/02    21.37
2023/03    21.70
2023/04    22.74
2023/05    20.48
2023/06    20.32
2023/07    20.69
2023/08    20.53
2023/09    20.38
2023/10    19.83
2023/11    21.39
2023/12    21.61


In [25]:
def tratar_grau_instrucao(df: pd.DataFrame) -> pd.DataFrame:
    """grau_de_instrucao vazio vira NÃO INFORMADO (a coluna é ~78% vazia em todos os meses)."""
    df = df.copy()
    df['grau_de_instrucao'] = df['grau_de_instrucao'].fillna('NÃO INFORMADO')
    return df

df = tratar_grau_instrucao(df)
print(df['grau_de_instrucao'].value_counts(dropna=False))

grau_de_instrucao
NÃO INFORMADO    632289
SUPERIOR         105448
MÉDIO             45767
BÁSICO            23473
Name: count, dtype: int64


### 4.8 "NAO DISPONIVEL" em outras colunas (só consulta)

In [26]:
# colunas de texto da tabela
colunas_texto = df.select_dtypes(include=['object', 'string']).columns

# em cada uma, quantas linhas têm exatamente "NAO DISPONIVEL"
contagem = {c: int((df[c] == 'NAO DISPONIVEL').sum()) for c in colunas_texto}

# mostra só as colunas onde o texto aparece
{c: n for c, n in contagem.items() if n > 0}

{'tipo_via': 973,
 'tipo_veiculo_vitima': 480,
 'tipo_de_vitima': 112828,
 'modo_transporte_vitima': 480,
 'sexo': 4366,
 'gravidade_lesao': 70314,
 'faixa_etaria_demografica': 8389,
 'faixa_etaria_legal': 8389,
 'local_obito': 105,
 'local_via': 117298}

### 4.9 "NAO DISPONIVEL" vira NÃO INFORMADO

In [27]:
COLUNAS_NAO_DISP = [
    'tipo_via', 'tipo_veiculo_vitima', 'tipo_de_vitima', 'modo_transporte_vitima',
    'sexo', 'faixa_etaria_demografica', 'faixa_etaria_legal', 'local_obito', 'local_via',
]

def tratar_nao_disponivel(df: pd.DataFrame, colunas: list) -> pd.DataFrame:
    """'NAO DISPONIVEL' é nulo disfarçado: vira 'NÃO INFORMADO', igual aos outros nulos."""
    df = df.copy()
    for c in colunas:
        df[c] = df[c].replace('NAO DISPONIVEL', 'NÃO INFORMADO')
    return df

df = tratar_nao_disponivel(df, COLUNAS_NAO_DISP)

# conferência: não deve sobrar nenhum
print({c: int((df[c] == 'NAO DISPONIVEL').sum()) for c in COLUNAS_NAO_DISP})

{'tipo_via': 0, 'tipo_veiculo_vitima': 0, 'tipo_de_vitima': 0, 'modo_transporte_vitima': 0, 'sexo': 0, 'faixa_etaria_demografica': 0, 'faixa_etaria_legal': 0, 'local_obito': 0, 'local_via': 0}


## 5. Diagnóstico DEPOIS (mesma tabela do passo 3)

In [28]:
depois = diagnostico(df)
depois

,dtype,pct_nulo,n_distintos
id_sinistro,int64,0.00,402695
id_pessoa,int64,0.00,806977
id_veiculo,Int64,21.04,554294
cod_ibge,int64,0.00,645
municipio,str,0.00,645
regiao_administrativa,str,0.00,16
tipo_via,str,0.00,3
tipo_veiculo_vitima,str,0.00,8
tipo_de_vitima,str,0.00,4
modo_transporte_vitima,str,0.00,9


## 6. Salvar a saída

In [29]:
df.to_parquet(SAIDA, index=False)
print('salvo em:', SAIDA, '| shape:', df.shape)

salvo em: c:\Users\pedro\Downloads\Projeto_integração_bd\data\intermediarios\05_nulos_categorias.parquet | shape: (806977, 34)


## 7. Resumo

**Nulos disfarçados:** o texto "NAO DISPONIVEL" aparecia em 9 colunas (`tipo_via`, `tipo_veiculo_vitima`, `tipo_de_vitima`, `modo_transporte_vitima`, `sexo`, `faixa_etaria_demografica`, `faixa_etaria_legal`, `local_obito` e `local_via`) e passou a ser `NÃO INFORMADO`, igual aos outros nulos. `gravidade_lesao` ficou como está, para a ETL-19.